In [15]:
pip install -U finance-datareader

In [16]:
# mq_quant.py

import FinanceDataReader as fdr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mticker
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

def get_close(ticker, start, end=None):
    """
    종가 데이터 수집 (Get Close Prices)
    """
    return fdr.DataReader(ticker, start, end)['Close']

def calc_daily_return(prices):
    """
    일별 수익률 계산 (Calculate Daily Returns)
    """
    return prices.pct_change().fillna(0)

def calc_cum_return(prices):
    """
    누적 수익률(자산 흐름) 계산 (Calculate Cumulative Returns)
    """
    return prices / prices.iloc[0]

def calc_portfolio(prices, weight=None):
    """
    포트폴리오 수익률 계산 (Calculate Portfolio Returns)
    returns: (일별 수익률, 누적 수익률)
    """
    # 1. 누적 수익률 계산
    cum_ret = calc_cum_return(prices)

    # 2. 비중 설정 (기본값: 동일 비중)
    if not weight:
        weight = [1/len(prices.columns)] * len(prices.columns)

    # 3. 포트폴리오 누적 수익률 (가중 평균)
    # 각 자산의 누적 수익률에 비중을 곱해 합산합니다. (Buy & Hold 가정)
    port_cum_ret = (cum_ret * weight).sum(axis=1)

    # 4. 포트폴리오 일별 수익률 (역산)
    port_daily_ret = port_cum_ret.pct_change().fillna(0)

    return port_daily_ret, port_cum_ret


def evaluate_performance(daily_ret, cum_ret, risk_free_rate=0.02):
    """
    성과 지표 평가 (CAGR, MDD, Volatility, Sharpe ratio)
    """

    # 1. 수익성 (CAGR)
    total_ret = cum_ret.iloc[-1]
    years = len(cum_ret) / 252
    cagr = total_ret ** (1 / years) - 1

    # 2. 안정성 (MDD)
    historical_max = cum_ret.cummax()
    dd = (cum_ret - historical_max) / historical_max * 100
    mdd = dd.min()

    # 3. 위험 (변동성)
    daily_vol = daily_ret.std()               # 일일 변동성
    annual_vol = daily_vol * np.sqrt(252)     # 연간 변동성

    # 4. 효율 (Sharpe Ratio)
    annual_return = daily_ret.mean() * 252    # 연간 수익률 (산술 평균)
    sharpe_ratio = (annual_return - risk_free_rate) / annual_vol # 위험 한 단위당 수익

    print(f"▶ 최종 수익률   : {(cum_ret.iloc[-1]-1)*100:.2f}%")
    print(f"=== 성과 평가 리포트 ===")
    print(f"1. 수익성 (CAGR) : {cagr*100:.2f}%")
    print(f"2. 안정성 (MDD)  : {mdd:.2f}%")
    print(f"-" * 30)
    print(f"3. 연간 변동성   : {annual_vol*100:.2f}%")
    print(f"4. 샤프 지수     : {sharpe_ratio:.4f}")

    return cagr, dd, mdd

def get_rebalancing_dates(close_data, period="month"):
    """
    리밸런싱 날짜 추출 (Pandas Grouper 활용)
    - 입력: 종가 데이터, 주기('month', 'quarter', 'year')
    - 출력: 리밸런싱 시행일(DatetimeIndex)
    """
    # 주기별 Pandas Frequency 문자열 매핑
    freq_map = {'month': 'ME', 'quarter': 'QE', 'year': 'YE'}

    # 1. 예외처리: 잘못된 주기 입력
    if period not in freq_map:
        raise ValueError("period must be 'month', 'quarter', or 'year'")

    freq = freq_map[period]

    rebalancing_dates = close_data.groupby(pd.Grouper(freq=freq)).apply(lambda x: x.index[-1])

    return rebalancing_dates.sort_index()


def cal_rebalancing_portfolio(close_data, period="month", weight_df=None, enable_plot=True):
    """
    리밸런싱 포트폴리오 성과 계산 (Chunk 방식)
    """

    # 1. 리밸런싱 날짜 구하기 (전체 데이터 기준)
    rebal_dates = get_rebalancing_dates(close_data, period)

    # 날짜 동기화 (weight_df가 있다면, 교집합 날짜만 사용)
    # close_data가 더 길어도, weight_df가 있는 기간만 백테스팅 수행
    if weight_df is not None:
        # weight_df 인덱스에 포함된 날짜만 필터링
        rebal_dates = rebal_dates[rebal_dates.isin(weight_df.index)]

        # 만약 겹치는 날짜가 하나도 없다면 에러 처리
        if rebal_dates.empty:
            print("Error: close_data와 weight_df의 리밸런싱 날짜가 일치하지 않습니다.")
            return None, None

    # 3. 초기 비중 설정 (없으면 동일 비중)
    if weight_df is None:
        n_assets = len(close_data.columns)
        weight_df = pd.DataFrame(
            index=rebal_dates,
            columns=close_data.columns,
            data=1/n_assets
        )

    # 4. 데이터 범위 보정 (시작일 기준)
    first_date = rebal_dates[0] # 필터링된 첫 날짜


    # 전체 기간 수익률 계산
    full_daily_rets = close_data.pct_change().fillna(0)

    # 백테스트 시작일 이후 데이터만 슬라이싱
    daily_rets = full_daily_rets.loc[first_date:]

    portfolio_chunks = []
    total_value = 1.0

    # 리밸런싱 기간별 순회
    full_dates = list(rebal_dates)

    # 마지막 구간 처리: 마지막 리밸런싱 날짜 ~ 데이터 끝 날짜
    # (단, daily_rets의 마지막 날짜가 리밸런싱 날짜보다 뒤에 있을 때만)
    if full_dates[-1] < daily_rets.index[-1]:
        full_dates.append(daily_rets.index[-1])

    for start, end in zip(full_dates[:-1], full_dates[1:]):

        # start가 weight_df에 없으면 건너뜀 (안전 장치)
        if start not in weight_df.index:
            continue

        current_weights = weight_df.loc[start]

        # start 다음날부터 end까지의 수익률 사용 (start 당일은 리밸런싱 날)
        chunk_rets = daily_rets.loc[start:end].iloc[1:]

        if chunk_rets.empty: continue

        cum_growth = (1 + chunk_rets).cumprod()
        chunk_value = (cum_growth * current_weights).sum(axis=1) * total_value
        portfolio_chunks.append(chunk_value)
        total_value = chunk_value.iloc[-1]

    # 6. 결과 병합
    if not portfolio_chunks:
        return None, None

    portfolio_cum_ret = pd.concat(portfolio_chunks)
    portfolio_cum_ret.loc[first_date] = 1.0 # 시작점 1.0 강제 할당
    portfolio_cum_ret = portfolio_cum_ret.sort_index()

    portfolio_day_ret = portfolio_cum_ret.pct_change().fillna(0)

    if enable_plot:
        historical_max = portfolio_cum_ret.cummax()
        dd = (portfolio_cum_ret - historical_max) / historical_max * 100
        mdd = dd.min()

        # [수정] 전체 컬럼이 아니라, weight_df에 존재하는(실제 투자된) 티커만 추출
        if weight_df is not None:
            # weight_df의 컬럼 중 close_data에도 존재하는 것만 리스트로 만듦 (KeyError 방지)
            tickers = [col for col in weight_df.columns if col in close_data.columns]

            if 'cash' in tickers: tickers.remove('cash')
        else:
            tickers = close_data.columns.tolist()

        plot_daily_rets = full_daily_rets.loc[portfolio_cum_ret.index]

        plot_portfolio_result(tickers, plot_daily_rets, portfolio_cum_ret, dd, mdd)

    return portfolio_day_ret, portfolio_cum_ret


def plot_portfolio_result(tickers, daily_rets, port_cum_ret, dd, mdd):
    """
    포트폴리오 성과 시각화 함수 (백테스트 기간 일치 버전)
    """

    plt.figure(figsize=(10, 8))

    # [차트 1] 포트폴리오 누적 수익률 (Asset Growth)
    plt.subplot(2, 1, 1)

    # 1. 메인 포트폴리오 선 그리기
    plt.plot(port_cum_ret.index, port_cum_ret, label='Portfolio', color='#d62728', linewidth=1.5)

    # 개별 종목 흐름 그리기 (기간 매칭 및 리베이스)
    if daily_rets is not None and tickers is not None:

        # 포트폴리오의 시작/종료 날짜 구하기
        start_date = port_cum_ret.index[0]
        end_date = port_cum_ret.index[-1]

        # 전체 데이터(daily_rets)를 실제 백테스트 기간만큼만 자르기
        subset_daily_rets = daily_rets.loc[start_date:end_date]

        for t in tickers:
            if t in subset_daily_rets.columns:
                # 잘라낸 기간의 수익률로 누적 수익률 새로 계산
                # 이렇게 해야 시작점이 1.0(혹은 포트폴리오 시작점) 근처로 맞춰집니다.
                ind_cum = (1 + subset_daily_rets[t]).cumprod()

                # 시각화 (투명도 조절)
                plt.plot(ind_cum.index, ind_cum, label=f'{t}', alpha=0.3, linewidth=0.8, linestyle='--')

    # Y축 로그 스케일 및 포맷 설정
    plt.yscale('log')
    ax = plt.gca()
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.yaxis.set_minor_formatter(mticker.ScalarFormatter())
    ax.ticklabel_format(style='plain', axis='y')

    plt.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
    plt.title('Portfolio Cumulative Return', fontsize=14, fontweight='bold')

    plt.legend(loc='upper left', fontsize='small')
    plt.grid(True, alpha=0.3)


    # [차트 2] 낙폭 (Drawdown)
    plt.subplot(2, 1, 2)

    plt.fill_between(dd.index, dd, 0, color='#1f77b4', alpha=0.3)
    plt.plot(dd.index, dd, color='#1f77b4', linewidth=0.5)

    plt.title('Portfolio Drawdown (MDD)', fontsize=12)
    plt.axhline(mdd, color='red', linestyle='--', label=f'Max DD: {mdd:.2f}%')

    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylabel('Drawdown (%)')

    plt.tight_layout()
    plt.show()

# 유니버스 구성(데이터 전처리)

In [17]:
# 데이터 로드
df = pd.read_csv('퀀트데이터_260102.csv')
df

,코드,회사명,시장구분,업종대,업종소,주가(원),시가총액(억),시가총액(억) 우선주포함,주식수 (단위:만주),유통주식 비중 (%),...,EPS 24년1Q,EPS 24년2Q,EPS 24년3Q,EPS 24년4Q,EPS 25년1Q,EPS 25년2Q,EPS 25년3Q(E),25년3Q 매출액,25년3Q 영업이익,25년3Q 지배순이익
0,A005930,삼성전자,코스피,반도체 관련장비 및 부품,종합 반도체,128500,7606735,8377015,591964,75.7,...,1109.09,1615.24,1638.51,1269.08,1356.23,833.50,2028.24,860617.47,121660.62,120064.61
1,A000660,SK하이닉스,코스피,반도체 관련장비 및 부품,종합 반도체,677000,4928576,4928576,72800,74.1,...,2636.33,5659.71,7896.51,10989.63,11136.06,9611.55,17301.04,244489.29,113833.90,125951.95
2,A373220,LG에너지솔루션,코스피,IT 장비 및 소재,2차전지,361000,844740,844740,23400,20.2,...,-1.98,-2016.12,569.04,-2904.53,-622.82,-1271.14,1056.32,56998.73,2358.13,2471.80
3,A207940,삼성바이오로직스,코스피,제약 및 바이오,바이오,1683000,779077,779077,4629,25.6,...,3708.86,6575.48,5468.71,6647.88,7765.74,6707.69,12409.76,16602.21,7288.48,5744.60
4,A005380,현대차,코스피,자동차 및 관련부품,완성차,298500,611202,739439,20476,65.9,...,15274.08,18956.66,14543.14,10889.03,15076.67,14643.15,11044.04,467214.48,25372.66,22613.53
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,A442900,미래에셋드림스팩1호,코스닥,금융,스팩,0,0,0,0,87.7,...,57.62,57.57,57.43,58.66,51.67,47.24,NaN,NaN,NaN,NaN
2696,A448370,하나27호스팩,코스닥,금융,스팩,0,0,0,0,94.7,...,7.41,10.44,11.10,11.37,8.79,8.24,NaN,NaN,NaN,NaN
2697,A448760,IBKS제22호스팩,코스닥,금융,스팩,0,0,0,0,91.0,...,8.10,7.63,9.75,11.15,3.29,0.57,NaN,0.00,-0.30,0.07
2698,A454750,하나28호스팩,코스닥,금융,스팩,0,0,0,0,100.0,...,13.05,11.91,14.89,11.40,8.27,5.37,NaN,NaN,NaN,NaN


In [19]:
# 시가총액 1000억 이상(유동성)
cond_cap = df['시가총액(억)'] >= 1000
cond_cap
# count = cond_cap.sum()
# print(count)

,시가총액(억)
0,True
1,True
2,True
3,True
4,True
...,...
2695,False
2696,False
2697,False
2698,False


In [20]:
# 거래대금 상위 500개
# 거래대금 컬럼이 여러 개이므로, 변동성을 줄이기 위해 '5일평균'을 사용합니다.
cond_liq = df['거래대금 (5일평균 억)'] >= df['거래대금 (5일평균 억)'].nlargest(500).min()
cond_liq

,거래대금 (5일평균 억)
0,True
1,True
2,True
3,True
4,True
...,...
2695,False
2696,False
2697,False
2698,False


In [6]:
# 안전성 필터링 (적자기업 추가)
cond_safe = (
    (df['관리종목 =1'] != 1) &
    (df['스팩 =1'] != 1) &
    (df['리츠 =1'] != 1) &
    (df['적자기업 =1'] != 1)
)

# 최종 유니버스 생성
universe = df[cond_cap & cond_liq & cond_safe].copy()
print(universe)
print(f"필터링 전: {len(df)}개 -> 필터링 후: {len(universe)}개")

           코드       회사명 시장구분            업종대     업종소    주가(원)  시가총액(억)  \
0     A005930      삼성전자  코스피  반도체 관련장비 및 부품  종합 반도체   128500  7606735   
1     A000660    SK하이닉스  코스피  반도체 관련장비 및 부품  종합 반도체   677000  4928576   
3     A207940  삼성바이오로직스  코스피       제약 및 바이오     바이오  1683000   779077   
4     A005380       현대차  코스피     자동차 및 관련부품     완성차   298500   611202   
5     A329180   HD현대중공업  코스피             조선      조선   504000   529005   
...       ...       ...  ...            ...     ...      ...      ...   
1389  A059270  해성에어로보틱스  코스닥             기계      기계    10040     1121   
1420  A071670   에이테크솔루션  코스닥             기계      금형    10770     1077   
1446  A011150     CJ씨푸드  코스피            식음료     식료품     2880     1035   
1447  A038460    바이오스마트  코스닥          IT서비스    전자결제     3955     1035   
1458  A452400       이닉스  코스닥     IT 장비 및 소재    2차전지    11240     1020   

      시가총액(억) 우선주포함  주식수 (단위:만주)  유통주식 비중 (%)  ...  EPS 24년1Q  EPS 24년2Q  \
0           8377015       591964         75.7  

# 팩터 스코어링

## 1)  저벨류 성장

In [7]:
# 컬럼명 매핑
col_per = 'PER'                   # 낮을수록 좋음
col_pbr = 'PBR'                   # 낮을수록 좋음
col_sales_yoy = '매출액 25년3Q(E) YOY'  # 높을수록 좋음
col_op_yoy = '영업이익 25년3Q(E) YOY'   # 높을수록 좋음
col_net_yoy = '지배순이익 25년3Q(E) YOY'# 높을수록 좋음

In [21]:
# 팩터 데이터 전처리 (결측치 제거 및 에러 방지)
score_cols = [col_per, col_pbr, col_sales_yoy, col_op_yoy, col_net_yoy]
universe = universe.dropna(subset=score_cols).copy()
universe

,코드,회사명,시장구분,업종대,업종소,주가(원),시가총액(억),시가총액(억) 우선주포함,주식수 (단위:만주),유통주식 비중 (%),...,rank_pbr,rank_sales,rank_op,rank_net,score_per,score_pbr,score_sales,score_op,score_net,total_score
0,A005930,삼성전자,코스피,반도체 관련장비 및 부품,종합 반도체,128500,7606735,8377015,591964,75.7,...,133.0,166.0,141.0,189.0,54.426230,56.721311,45.901639,54.098361,38.360656,50.459016
1,A000660,SK하이닉스,코스피,반도체 관련장비 및 부품,종합 반도체,677000,4928576,4928576,72800,74.1,...,232.0,61.0,103.0,104.0,74.754098,24.262295,80.327869,66.557377,66.229508,60.032787
3,A207940,삼성바이오로직스,코스피,제약 및 바이오,바이오,1683000,779077,779077,4629,25.6,...,257.0,59.0,73.0,106.0,29.180328,16.065574,80.983607,76.393443,65.573770,48.032787
4,A005380,현대차,코스피,자동차 및 관련부품,완성차,298500,611202,739439,20476,65.9,...,26.0,165.0,248.0,254.0,96.065574,91.803279,46.229508,19.016393,17.049180,60.163934
5,A329180,HD현대중공업,코스피,조선,조선,504000,529005,529005,10496,30.6,...,279.0,106.0,53.0,33.0,34.754098,8.852459,65.573770,82.950820,89.508197,51.081967
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1389,A059270,해성에어로보틱스,코스닥,기계,기계,10040,1121,1121,1117,66.7,...,178.0,144.0,87.0,147.0,2.622951,41.967213,53.114754,71.803279,52.131148,40.311475
1420,A071670,에이테크솔루션,코스닥,기계,금형,10770,1077,1077,1000,66.3,...,85.0,213.0,286.0,78.0,8.852459,72.459016,30.491803,6.557377,74.754098,40.836066
1446,A011150,CJ씨푸드,코스피,식음료,식료품,2880,1035,1066,3593,53.5,...,86.0,265.0,301.0,291.0,27.213115,72.131148,13.442623,1.639344,4.918033,28.081967
1447,A038460,바이오스마트,코스닥,IT서비스,전자결제,3955,1035,1035,2616,71.9,...,43.0,192.0,77.0,44.0,72.786885,86.229508,37.377049,75.081967,85.901639,73.803279


In [22]:
# 랭킹 및 스코어링 계산 (0~100점 변환)
# method='min': 동점자 발생 시 최소 등수 부여 (예: 공동 1등)

# N = 유니버스 개수
N = len(universe)

# [저평가 팩터] 낮을수록 좋은 지표 (ascending=True -> 작은 값이 1등)
universe['rank_per'] = universe[col_per].rank(ascending=True, method='min')
universe['rank_pbr'] = universe[col_pbr].rank(ascending=True, method='min')

# [성장성 팩터] 높을수록 좋은 지표 (ascending=False -> 큰 값이 1등)
universe['rank_sales'] = universe[col_sales_yoy].rank(ascending=False, method='min')
universe['rank_op'] = universe[col_op_yoy].rank(ascending=False, method='min')
universe['rank_net'] = universe[col_net_yoy].rank(ascending=False, method='min')

In [24]:
# 랭킹을 점수(0~100)로 변환하는 함수
# 1등 -> 100점, 꼴등(N등) -> 0점
def rank_to_score(rank, n):
    if n <= 1: return 100.0
    return ((n - rank) / (n - 1)) * 100

# 각 지표별 점수 계산
universe['score_per'] = universe['rank_per'].apply(lambda r: rank_to_score(r, N))
universe['score_pbr'] = universe['rank_pbr'].apply(lambda r: rank_to_score(r, N))
universe['score_sales'] = universe['rank_sales'].apply(lambda r: rank_to_score(r, N))
universe['score_op'] = universe['rank_op'].apply(lambda r: rank_to_score(r, N))
universe['score_net'] = universe['rank_net'].apply(lambda r: rank_to_score(r, N))



In [25]:
# 가중치 적용하여 최종 점수 계산 (총합 100점 만점 기준)
# 비중: PER 25, PBR 25, 매출 15, 영업익 15, 순익 20
w_per = 0.25
w_pbr = 0.25
w_sales = 0.15
w_op = 0.15
w_net = 0.20

universe['total_score'] = (
    (universe['score_per'] * w_per) +
    (universe['score_pbr'] * w_pbr) +
    (universe['score_sales'] * w_sales) +
    (universe['score_op'] * w_op) +
    (universe['score_net'] * w_net)
)

In [26]:
#정렬 및 통합순위 생성
# 점수가 높은 순서대로 정렬 (100점이 1등)
final_ranking = universe.sort_values('total_score', ascending=False).copy()

# 1등부터 번호 매기기 (range 함수 사용)
final_ranking['통합순위'] = range(1, len(final_ranking) + 1)

In [30]:
from tabulate import tabulate

# 결과 확인 및 저장
output_cols = ['코드', '통합순위', '회사명', 'total_score', 'PER', 'PBR', '매출액 25년3Q(E) YOY', '영업이익 25년3Q(E) YOY', '지배순이익 25년3Q(E) YOY', ]

# 상위 20개 출력
print_df = final_ranking[output_cols].head(20)
# (headers='keys', tablefmt='psql')
print(tabulate(print_df, headers='keys', tablefmt='psql', floatfmt=".2f"))

+------+---------+------------+------------------+---------------+-------+-------+------------------------+--------------------------+----------------------------+
|      | 코드    |   통합순위 | 회사명           |   total_score |   PER |   PBR |   매출액 25년3Q(E) YOY |   영업이익 25년3Q(E) YOY |   지배순이익 25년3Q(E) YOY |
|------+---------+------------+------------------+---------------+-------+-------+------------------------+--------------------------+----------------------------|
|  148 | A088350 |          1 | 한화생명         |         91.36 |  3.63 |  0.22 |                  23.82 |                   176.87 |                    1053.22 |
|  154 | A001450 |          2 | 현대해상         |         83.93 |  2.60 |  0.53 |                  18.37 |                   139.37 |                     196.69 |
|  739 | A033530 |          3 | SJG세종          |         83.54 |  4.31 |  0.56 |                   6.57 |                    79.26 |                    4792.53 |
|  353 | A006730 |          4 | 서부T&D          |  

In [28]:
# CSV 저장
final_ranking[output_cols].to_csv('저벨류성장_260104.csv', index=False, encoding='utf-8-sig')

## 2) 모멘텀 전략

In [54]:
# 데이터 로드
df = pd.read_csv('퀀트데이터_260102.csv')

# 시가총액 1000억 이상(유동성)
cond_cap = df['시가총액(억)'] >= 1000
cond_cap
cond_liq = df['거래대금 (5일평균 억)'] >= df['거래대금 (5일평균 억)'].nlargest(500).min()
cond_liq

# 기본 필터링 (관리종목, 스팩, 리츠, 적자기업 제외)
mask = (df['관리종목 =1'] != 1) & (df['스팩 =1'] != 1) & (df['리츠 =1'] != 1) & (df['적자기업 =1'] != 1) & (cond_cap) & (cond_liq)
target_df = df[mask].copy()
print(target_df)
print(f"필터링 전: {len(df)}개 -> 필터링 후: {len(target_df)}개")


           코드       회사명 시장구분            업종대     업종소    주가(원)  시가총액(억)  \
0     A005930      삼성전자  코스피  반도체 관련장비 및 부품  종합 반도체   128500  7606735   
1     A000660    SK하이닉스  코스피  반도체 관련장비 및 부품  종합 반도체   677000  4928576   
3     A207940  삼성바이오로직스  코스피       제약 및 바이오     바이오  1683000   779077   
4     A005380       현대차  코스피     자동차 및 관련부품     완성차   298500   611202   
5     A329180   HD현대중공업  코스피             조선      조선   504000   529005   
...       ...       ...  ...            ...     ...      ...      ...   
1389  A059270  해성에어로보틱스  코스닥             기계      기계    10040     1121   
1420  A071670   에이테크솔루션  코스닥             기계      금형    10770     1077   
1446  A011150     CJ씨푸드  코스피            식음료     식료품     2880     1035   
1447  A038460    바이오스마트  코스닥          IT서비스    전자결제     3955     1035   
1458  A452400       이닉스  코스닥     IT 장비 및 소재    2차전지    11240     1020   

      시가총액(억) 우선주포함  주식수 (단위:만주)  유통주식 비중 (%)  ...  EPS 24년1Q  EPS 24년2Q  \
0           8377015       591964         75.7  

In [59]:
# 랭킹 산출 (낮은 순위 = 1등)
# 1개월: 낮을수록 좋음 (ascending=True)
target_df['R_1m'] = target_df['1개월 등락률 (%)'].rank(ascending=True)


In [60]:
# 3,6,9,12개월: 높을수록 좋음 (ascending=False)
target_df['R_3m'] = target_df['3개월 등락률 (%)'].rank(ascending=False)
target_df['R_6m'] = target_df['6개월 등락률 (%)'].rank(ascending=False)
target_df['R_9m'] = target_df['9개월 등락률 (%)'].rank(ascending=False)
target_df['R_12m'] = target_df['1년 등락률 (%)'].rank(ascending=False)

In [61]:
# 랭킹을 점수로 변환하는 함수 정의
def rank_to_score(rank, n):
    if n <= 1: return 100.0
    return ((n - rank) / (n - 1)) * 100

N = len(target_df)

#  랭킹 -> 점수 변환 (0~100점)
# 1등(Rank 1)일수록 100점에 가깝게 변환됨
target_df['S_1m'] = target_df['R_1m'].apply(lambda r: rank_to_score(r, N))
target_df['S_3m'] = target_df['R_3m'].apply(lambda r: rank_to_score(r, N))
target_df['S_6m'] = target_df['R_6m'].apply(lambda r: rank_to_score(r, N))
target_df['S_9m'] = target_df['R_9m'].apply(lambda r: rank_to_score(r, N))
target_df['S_12m'] = target_df['R_12m'].apply(lambda r: rank_to_score(r, N))

# 5. 종합 점수 계산 (각 20% 비중)
target_df['Total_Score'] = (
    0.2 * target_df['S_1m'] +
    0.2 * target_df['S_3m'] +
    0.2 * target_df['S_6m'] +
    0.2 * target_df['S_9m'] +
    0.2 * target_df['S_12m']
)

In [62]:
# 음수 보정 (절대 모멘텀 필터)
# 1년 수익률이 마이너스인 종목은 추세가 꺾인 것으로 보고 제외
final_df = target_df[target_df['1년 등락률 (%)'] > 0].copy()

In [63]:
from tabulate import tabulate

# 결과 출력
m_cols = ['회사명', '1개월 등락률 (%)', '3개월 등락률 (%)', '6개월 등락률 (%)', '9개월 등락률 (%)', '1년 등락률 (%)', 'Total_Score']

# final_df에서 Total_Score를 기준으로 정렬하여 상위 20개 출력
print_df = final_df.sort_values('Total_Score', ascending=False).head(20)

print(tabulate(print_df[m_cols], headers='keys', tablefmt='psql', floatfmt=".2f"))

+------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+---------------+
|      | 회사명           |   1개월 등락률 (%) |   3개월 등락률 (%) |   6개월 등락률 (%) |   9개월 등락률 (%) |   1년 등락률 (%) |   Total_Score |
|------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+---------------|
|  236 | 씨어스테크놀로지 |              -4.65 |              76.58 |             362.90 |             876.95 |           971.49 |         94.97 |
|   70 | 이수페타시스     |             -11.70 |              58.20 |             147.20 |             272.13 |           355.12 |         93.31 |
| 1089 | 씨피시스템       |              -9.31 |              64.73 |             245.80 |             228.98 |           151.67 |         91.59 |
|  691 | 한국피아이엠     |             -19.38 |              94.72 |             259.58 |             220.50 |            99.61 |         90.70 |
|  120 | 로보티즈         |

In [64]:
# CSV 저장
final_df[m_cols].to_csv('모멘텀_260104.csv', index=False, encoding='utf-8-sig')

실습: PEG 개념을 이용하여 저평가 성장주 구성